In [4]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

# ==========================================
# 0. GOOGLE COLAB FILE UPLOAD HELPER
# ==========================================
BASE_DIR = Path.cwd()
EXPECTED_FILE = BASE_DIR / "qr_verification_dataset.csv"

if not EXPECTED_FILE.exists():
    try:
        from google.colab import files
        print("Select 'qr_verification_dataset.csv' from C:\\Users\\alexn\\Downloads:")
        uploaded = files.upload()
        if uploaded:
            EXPECTED_FILE = BASE_DIR / next(iter(uploaded.keys()))
    except ImportError:
        pass

# ==========================================
# 1. CONFIGURATION & FILE PATHS
# ==========================================
ARTIFACT_DIR = BASE_DIR / "artifacts"
MODEL_OUTPUT_PATH = ARTIFACT_DIR / "isolation_forest_qr.joblib"
REPORT_OUTPUT_PATH = ARTIFACT_DIR / "evaluation_report.json"

FEATURE_COLUMNS = [
    "hour_of_day",
    "is_outside_operating_hours",
    "log_time_since_last_scan",
    "spatial_order_diff",
    "impossible_travel_flag",
    "rapid_same_building_repeat",
    "scans_last_10min",
    "scans_last_1hr",
    "failed_scans_last_10min",
    "result_encoded",
]

CAMPUS_TOPOLOGY = {
    "Central Building": {"campus_phase": "Phase 1", "phase_code": 1, "phase2_order": 0},
    "University Library": {"campus_phase": "Phase 2", "phase_code": 2, "phase2_order": 1},
    "Auditorium": {"campus_phase": "Phase 2", "phase_code": 2, "phase2_order": 2},
    "Oval Building": {"campus_phase": "Phase 2", "phase_code": 2, "phase2_order": 3},
    "STMB": {"campus_phase": "Phase 2", "phase_code": 2, "phase2_order": 4},
    "MSB": {"campus_phase": "Phase 2", "phase_code": 2, "phase2_order": 5},
    "Forge": {"campus_phase": "Phase 2", "phase_code": 2, "phase2_order": 6},
}


def load_dataset(file_path: Path) -> tuple[pd.DataFrame, pd.Series]:
    if not file_path.exists():
        matches = sorted(BASE_DIR.glob("qr_verification*.*"))
        if matches:
            file_path = matches[0]
        else:
            raise FileNotFoundError(
                f"Dataset not found in '{BASE_DIR}'. Upload 'qr_verification_dataset.csv' and re-run."
            )

    print(f"Loading dataset from: {file_path}")
    if file_path.suffix.lower() in [".xlsx", ".xls"]:
        xls = pd.ExcelFile(file_path)
        df = pd.read_excel(xls, sheet_name=xls.sheet_names[0])
    else:
        df = pd.read_csv(file_path)

    df.columns = df.columns.str.strip()
    missing_cols = [c for c in FEATURE_COLUMNS + ["is_anomaly"] if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in dataset: {missing_cols}")

    return df[FEATURE_COLUMNS].copy(), df["is_anomaly"].astype(int).copy()


def evaluate_predictions(y_true: pd.Series, y_pred: np.ndarray, anomaly_scores: np.ndarray) -> dict:
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "f1_score": round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(y_true, anomaly_scores)), 4),
    }


def run_cross_validation(X_train: pd.DataFrame, y_train: pd.Series, params: dict, n_splits: int = 5) -> dict:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_accs, val_accs, val_f1s, val_aucs = [], [], [], []

    for fold_train_idx, fold_val_idx in skf.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[fold_train_idx], y_train.iloc[fold_train_idx]
        X_va, y_va = X_train.iloc[fold_val_idx], y_train.iloc[fold_val_idx]

        clf = IsolationForest(**params)
        clf.fit(X_tr)

        pred_tr = np.where(clf.predict(X_tr) == -1, 1, 0)
        pred_va = np.where(clf.predict(X_va) == -1, 1, 0)
        score_va = -clf.decision_function(X_va)

        train_accs.append(accuracy_score(y_tr, pred_tr))
        val_accs.append(accuracy_score(y_va, pred_va))
        val_f1s.append(f1_score(y_va, pred_va, zero_division=0))
        val_aucs.append(roc_auc_score(y_va, score_va))

    return {
        "cv_train_accuracy_mean": round(float(np.mean(train_accs)), 4),
        "cv_val_accuracy_mean": round(float(np.mean(val_accs)), 4),
        "cv_val_accuracy_std": round(float(np.std(val_accs)), 4),
        "cv_val_f1_mean": round(float(np.mean(val_f1s)), 4),
        "cv_val_roc_auc_mean": round(float(np.mean(val_aucs)), 4),
    }


def main():
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    X, y = load_dataset(EXPECTED_FILE)

    # 1. Stratified 80/20 Train-Test Split (prevents data leakage)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42
    )

    # 2. Regularized Hyperparameters (prevents overfitting & underfitting)
    model_params = {
        "n_estimators": 200,
        "max_samples": 256,
        "max_features": 0.7,
        "contamination": float(y_train.mean()),
        "random_state": 42,
        "n_jobs": -1,
    }

    # 3. 5-Fold Cross-Validation on Training Set
    cv_summary = run_cross_validation(X_train, y_train, model_params, n_splits=5)

    # 4. Fit Final Model Strictly on Training Split
    model = IsolationForest(**model_params)
    model.fit(X_train)

    # 5. Evaluate on Train vs. Unseen Test Split
    y_train_pred = np.where(model.predict(X_train) == -1, 1, 0)
    train_scores = -model.decision_function(X_train)
    train_metrics = evaluate_predictions(y_train, y_train_pred, train_scores)

    y_test_pred = np.where(model.predict(X_test) == -1, 1, 0)
    test_scores = -model.decision_function(X_test)
    test_metrics = evaluate_predictions(y_test, y_test_pred, test_scores)

    acc_gap = abs(train_metrics["accuracy"] - test_metrics["accuracy"])
    fit_status = "OPTIMAL FIT"
    if test_metrics["accuracy"] < 0.85:
        fit_status = "UNDERFITTING (Below 85% target accuracy)"
    elif acc_gap > 0.05:
        fit_status = "OVERFITTING WARNING (Train/Test accuracy gap > 5%)"

    print("=" * 55)
    print("ISOLATION FOREST TRAINING & GENERALIZATION REPORT")
    print("=" * 55)
    print(f"5-Fold CV Train Accuracy : {cv_summary['cv_train_accuracy_mean']:.4f}")
    print(f"5-Fold CV Val Accuracy   : {cv_summary['cv_val_accuracy_mean']:.4f} (+/- {cv_summary['cv_val_accuracy_std']:.4f})")
    print(f"Holdout Train Accuracy   : {train_metrics['accuracy']:.4f}")
    print(f"Holdout Test Accuracy    : {test_metrics['accuracy']:.4f}")
    print(f"Generalization Gap       : {acc_gap:.4f} -> Status: {fit_status}\n")

    print("--- Held-Out Test Set Metrics (Sprint 5 Evaluation) ---")
    for metric_name, val in test_metrics.items():
        print(f"{metric_name.upper():<12}: {val:.4f}")

    print("\nConfusion Matrix (Test Set):")
    print(confusion_matrix(y_test, y_test_pred))

    print("\nClassification Report (Test Set):")
    print(classification_report(y_test, y_test_pred, target_names=["Normal (0)", "Anomaly (1)"]))

    # 6. Save Model Artifact & Evaluation Report
    artifact_bundle = {
        "model": model,
        "feature_columns": FEATURE_COLUMNS,
        "campus_topology": CAMPUS_TOPOLOGY,
        "hyperparameters": model_params,
        "test_metrics": test_metrics,
    }
    joblib.dump(artifact_bundle, MODEL_OUTPUT_PATH)

    report_payload = {
        "status": fit_status,
        "generalization_gap": round(acc_gap, 4),
        "cross_validation": cv_summary,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
    }
    with open(REPORT_OUTPUT_PATH, "w") as f:
        json.dump(report_payload, f, indent=2)

    print(f"\nSaved trained model to : {MODEL_OUTPUT_PATH}")
    print(f"Saved metrics report to: {REPORT_OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Select 'qr_verification_dataset.csv' from C:\Users\alexn\Downloads:


Saving qr_verification_dataset.csv to qr_verification_dataset.csv
Loading dataset from: /content/qr_verification_dataset.csv
ISOLATION FOREST TRAINING & GENERALIZATION REPORT
5-Fold CV Train Accuracy : 0.9312
5-Fold CV Val Accuracy   : 0.9309 (+/- 0.0119)
Holdout Train Accuracy   : 0.9315
Holdout Test Accuracy    : 0.9496
Generalization Gap       : 0.0181 -> Status: OPTIMAL FIT

--- Held-Out Test Set Metrics (Sprint 5 Evaluation) ---
ACCURACY    : 0.9496
PRECISION   : 0.7895
RECALL      : 0.6383
F1_SCORE    : 0.7059
ROC_AUC     : 0.8118

Confusion Matrix (Test Set):
[[441   8]
 [ 17  30]]

Classification Report (Test Set):
              precision    recall  f1-score   support

  Normal (0)       0.96      0.98      0.97       449
 Anomaly (1)       0.79      0.64      0.71        47

    accuracy                           0.95       496
   macro avg       0.88      0.81      0.84       496
weighted avg       0.95      0.95      0.95       496


Saved trained model to : /content/artifac